# Prediction of Serum Estradiol Concentrations Using Machine Learning

**An exploratory analysis of the National Health and Nutrition Examination Survey (NHANES), 2021–2023**

## Abstract

**Background:** Serum estradiol concentrations vary in relation to reproductive physiology and other biological characteristics. The extent to which concurrent hormonal and demographic measurements can predict estradiol was investigated using publicly available NHANES data.

**Methods:** Hormone, demographic and anthropometric datasets were linked by participant identifier. Adult observations with complete measurements for eight predictors and estradiol concentrations at or above the assay detection limit were included in regression modelling. A median baseline, Ridge regression, Random Forest and histogram-based Gradient Boosting were evaluated. Model selection was informed by five-fold cross-validation of the training data. Prediction errors were additionally examined in relation to estradiol concentration and recorded pregnancy status.

**Results:** The modelling dataset comprised 5,131 participants, including 4,104 training and 1,027 test observations. Random Forest achieved a mean cross-validation R² of 0.788 and mean absolute error (MAE) of 0.377 on the log-transformed scale. In the exploratory test set, log-scale R² was 0.741, while original-scale R² was 0.899 and MAE was 17.24 pg/mL. Restriction of evaluation to concentrations below 200 pg/mL reduced original-scale R² to 0.677. The 20 highest measured estradiol concentrations occurred in participants recorded as pregnant.

**Conclusion:** Concurrent hormonal and demographic measurements contained predictive information about estradiol. Nevertheless, performance varied across concentration ranges, and the apparent strength of overall model fit was influenced by extreme values. The analysis is exploratory and does not constitute clinical validation.

## 1. Introduction

Estradiol is a steroid hormone involved in reproductive and broader physiological processes. Circulating concentrations are influenced by reproductive status, age and interactions among endocrine pathways. Understanding these relationships provides a biologically relevant setting for evaluating supervised machine learning methods.

The present study investigated whether serum estradiol concentrations could be estimated from demographic, anthropometric and other laboratory hormone measurements collected in NHANES 2021–2023. In addition to comparing regression algorithms, the analysis examined the distribution of prediction errors and their possible biological context.

**Research question:** To what extent can serum estradiol concentration (pg/mL) be predicted from demographic, anthropometric and reproductive hormone measurements in the analysed NHANES cohort?

**Data source:** [CDC NHANES 2021–2023](https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/default.aspx?Cycle=2021-2023). Public XPT files were retrieved from CDC endpoints during notebook execution.

**Reproducibility:** Code cells were designed to run sequentially in a fresh Python environment using the packages in `requirements.txt`. Execution requires access to the CDC source files.


## 2. Materials and Methods

### 2.1 Data sources and cohort construction

Publicly available NHANES hormone, demographic and body measurement files were imported and linked using the unique participant identifier (`SEQN`). The analytical cohort was restricted to adults aged 20 years or older. Descriptive checks were undertaken to assess the dimensions of the merged data and the availability of key variables.


In [ ]:

# Import the necessary libraries

import pandas as pd
import urllib.request

# Specify the official CDC dataset location

url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/TST_L.XPT"

# Download the dataset into Google Colab

urllib.request.urlretrieve(url, "TST_L.XPT")

# Load the downloaded dataset into Python

hormones = pd.read_sas("TST_L.XPT", format="xport")

# Confirm that the download worked

print("Dataset downloaded successfully!")

print("Number of rows and columns:", hormones.shape)

# Display the first five rows

display(hormones.head())

In [ ]:

# Download the NHANES demographics dataset

demo_url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.XPT"

demographics = pd.read_sas(
    demo_url,
    format="xport"
)

print("Demographics dataset downloaded!")

print("Rows and columns:", demographics.shape)

# Display the first five rows

display(demographics.head())

In [ ]:

# Download the NHANES body measurements dataset

bmx_url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.XPT"

body_measurements = pd.read_sas(
    bmx_url,
    format="xport"
)

print("Body measurements dataset downloaded!")

print("Rows and columns:", body_measurements.shape)

# Display the first five rows

display(body_measurements.head())

In [ ]:

# Merge hormone data with demographic data

combined_data = pd.merge(
    hormones,
    demographics[["SEQN", "RIAGENDR", "RIDAGEYR"]],
    on="SEQN",
    how="left"
)

# Add BMI measurements

combined_data = pd.merge(
    combined_data,
    body_measurements[["SEQN", "BMXBMI"]],
    on="SEQN",
    how="left"
)

print("Datasets successfully merged!")

print("Combined dataset dimensions:")
print(combined_data.shape)

# Display the first five rows

display(combined_data.head())

In [ ]:

# Create an adult-only dataset

adult_data = combined_data[
    combined_data["RIDAGEYR"] >= 20
].copy()

print("Adult dataset created!")

print("Number of adult participants:", len(adult_data))

# Display the first five rows

display(adult_data.head())

In [ ]:

# Check missing values in key variables

important_variables = [
    "LBXEST",
    "RIAGENDR",
    "RIDAGEYR",
    "BMXBMI"
]

print("MISSING DATA SUMMARY")
print("--------------------")

display(
    adult_data[important_variables].isnull().sum()
)

# Check the age range

print("\nAGE RANGE")

print("Minimum age:", adult_data["RIDAGEYR"].min())
print("Maximum age:", adult_data["RIDAGEYR"].max())

# Check gender counts

print("\nGENDER DISTRIBUTION")

print(
    adult_data["RIAGENDR"].value_counts()
)

### 2.2 Exploratory data analysis

Missing observations and the distribution of serum estradiol were examined using descriptive statistics and graphical methods. The marked right-skew of the concentration distribution was considered when choosing visualisation ranges. Differences by recorded sex and associations with age and body mass index (BMI) were examined descriptively. These summaries were unweighted and describe the analysed observations rather than nationally representative population estimates.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:

# Count missing values in every column

missing_values = hormones.isnull().sum()

# Calculate the percentage of missing values

missing_percentage = (
    hormones.isnull().mean() * 100
)

# Combine the results into a table

missing_summary = pd.DataFrame({
    "Missing values": missing_values,
    "Missing percentage (%)": missing_percentage
})

# Sort from most missing to least missing

missing_summary = missing_summary.sort_values(
    by="Missing percentage (%)",
    ascending=False
)

display(missing_summary)

In [ ]:

# Investigate estradiol concentrations in more detail

print("ESTRADIOL DISTRIBUTION ANALYSIS")
print("--------------------------------")

print("Minimum:", hormones["LBXEST"].min())
print("Maximum:", hormones["LBXEST"].max())
print("Median:", hormones["LBXEST"].median())
print("Mean:", hormones["LBXEST"].mean())

print("\nSelected percentiles:")

print(hormones["LBXEST"].quantile(
    [0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
))

In [ ]:

# Calculate the 99th percentile
upper_limit = hormones["LBXEST"].quantile(0.99)

# Select measurements within the plotting range
# This only filters the graph, not the original dataset

estradiol_plot = hormones.loc[
    hormones["LBXEST"].between(0, upper_limit),
    "LBXEST"
].dropna()

# Create the improved histogram

plt.figure(figsize=(10, 6))

sns.histplot(
    x=estradiol_plot,
    bins=40,
    kde=True
)

plt.title("Distribution of Serum Estradiol Concentrations (Central 99%)")
plt.xlabel("Estradiol concentration (pg/mL)")
plt.ylabel("Frequency")

plt.tight_layout()

# Save the corrected graph
plt.savefig(
    "estradiol_distribution_zoomed.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:

# Summary statistics for estradiol

print("MEAN:", hormones["LBXEST"].mean())
print("MEDIAN:", hormones["LBXEST"].median())
print("STANDARD DEVIATION:", hormones["LBXEST"].std())

print("\nPERCENTILES:")

display(
    hormones["LBXEST"].quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
    )
)

In [ ]:

# Compare estradiol concentrations between sexes

sex_summary = adult_data.groupby("RIAGENDR")["LBXEST"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)

# Replace numerical codes with readable labels

sex_summary.index = sex_summary.index.map({
    1.0: "Male",
    2.0: "Female"
})

# Display the results

print("ESTRADIOL STATISTICS BY SEX")
print("---------------------------")

display(sex_summary.round(2))

In [ ]:

# Prepare data for visualisation

plot_data = adult_data.dropna(
    subset=["LBXEST", "RIAGENDR"]
).copy()

# Convert numerical sex codes into labels

plot_data["Sex"] = plot_data["RIAGENDR"].map({
    1.0: "Male",
    2.0: "Female"
})

# Create boxplot

plt.figure(figsize=(9, 6))

sns.boxplot(
    data=plot_data,
    x="Sex",
    y="LBXEST"
)

plt.yscale("log")

plt.title("Serum Estradiol Concentrations by Sex")
plt.xlabel("Sex")
plt.ylabel("Estradiol concentration (pg/mL, logarithmic scale)")

plt.tight_layout()

plt.savefig(
    "estradiol_by_sex.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:

# Create a scatterplot of age against estradiol

age_plot_data = adult_data.dropna(
    subset=["RIDAGEYR", "LBXEST"]
)

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=age_plot_data,
    x="RIDAGEYR",
    y="LBXEST",
    alpha=0.35,
    s=20
)

plt.yscale("log")

plt.title("Relationship Between Age and Estradiol Concentration")
plt.xlabel("Age (years)")
plt.ylabel("Estradiol concentration (pg/mL, logarithmic scale)")

plt.tight_layout()

plt.savefig(
    "estradiol_vs_age.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:

# Import Spearman correlation

from scipy.stats import spearmanr

# Separate participants by sex

male_data = adult_data[
    adult_data["RIAGENDR"] == 1
].dropna(subset=["RIDAGEYR", "LBXEST"])

female_data = adult_data[
    adult_data["RIAGENDR"] == 2
].dropna(subset=["RIDAGEYR", "LBXEST"])

# Calculate correlations

male_corr, male_p = spearmanr(
    male_data["RIDAGEYR"],
    male_data["LBXEST"]
)

female_corr, female_p = spearmanr(
    female_data["RIDAGEYR"],
    female_data["LBXEST"]
)

# Display results

print("AGE AND ESTRADIOL CORRELATION")
print("----------------------------")

print("\nMALES")
print("Spearman correlation:", round(male_corr, 3))
print("P-value:", male_p)

print("\nFEMALES")
print("Spearman correlation:", round(female_corr, 3))
print("P-value:", female_p)

In [ ]:

# Create separate scatterplots for males and females

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Male scatterplot

sns.scatterplot(
    data=male_data,
    x="RIDAGEYR",
    y="LBXEST",
    alpha=0.4,
    ax=axes[0]
)

axes[0].set_yscale("log")
axes[0].set_title("Male Estradiol Concentrations")
axes[0].set_xlabel("Age (years)")
axes[0].set_ylabel("Estradiol (pg/mL)")

# Female scatterplot

sns.scatterplot(
    data=female_data,
    x="RIDAGEYR",
    y="LBXEST",
    alpha=0.4,
    ax=axes[1]
)

axes[1].set_yscale("log")
axes[1].set_title("Female Estradiol Concentrations")
axes[1].set_xlabel("Age (years)")
axes[1].set_ylabel("Estradiol (pg/mL)")

plt.suptitle(
    "Relationship Between Age and Estradiol by Sex",
    fontsize=15
)

plt.tight_layout()

plt.savefig(
    "estradiol_age_by_sex.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:

# Investigate the relationship between BMI and estradiol

from scipy.stats import spearmanr

# Prepare data for analysis

bmi_data = adult_data.dropna(
    subset=["BMXBMI", "LBXEST", "RIAGENDR"]
).copy()

# Separate males and females

male_bmi = bmi_data[bmi_data["RIAGENDR"] == 1]

female_bmi = bmi_data[bmi_data["RIAGENDR"] == 2]

# Calculate Spearman correlations

male_bmi_corr, male_bmi_p = spearmanr(
    male_bmi["BMXBMI"],
    male_bmi["LBXEST"]
)

female_bmi_corr, female_bmi_p = spearmanr(
    female_bmi["BMXBMI"],
    female_bmi["LBXEST"]
)

# Display results

print("BMI AND ESTRADIOL CORRELATION")
print("----------------------------")

print("\nMALES")
print("Participants:", len(male_bmi))
print("Spearman correlation:", round(male_bmi_corr, 3))
print("P-value:", male_bmi_p)

print("\nFEMALES")
print("Participants:", len(female_bmi))
print("Spearman correlation:", round(female_bmi_corr, 3))
print("P-value:", female_bmi_p)

In [ ]:

# Create BMI versus estradiol scatterplots

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Male participants

sns.scatterplot(
    data=male_bmi,
    x="BMXBMI",
    y="LBXEST",
    alpha=0.4,
    ax=axes[0]
)

axes[0].set_yscale("log")
axes[0].set_title("Male Participants")
axes[0].set_xlabel("BMI (kg/m²)")
axes[0].set_ylabel("Estradiol (pg/mL)")

# Female participants

sns.scatterplot(
    data=female_bmi,
    x="BMXBMI",
    y="LBXEST",
    alpha=0.4,
    ax=axes[1]
)

axes[1].set_yscale("log")
axes[1].set_title("Female Participants")
axes[1].set_xlabel("BMI (kg/m²)")
axes[1].set_ylabel("Estradiol (pg/mL)")

plt.suptitle(
    "Relationship Between BMI and Estradiol by Sex",
    fontsize=15
)

plt.tight_layout()

plt.savefig(
    "estradiol_vs_bmi.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:

# Summarise BMI measurements by sex

bmi_summary = bmi_data.groupby("RIAGENDR")["BMXBMI"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)

bmi_summary.index = bmi_summary.index.map({
    1.0: "Male",
    2.0: "Female"
})

print("BMI SUMMARY BY SEX")

display(bmi_summary.round(2))

### 2.3 Statistical analysis and assay considerations

Spearman rank correlation was used to quantify monotonic associations between estradiol and continuous variables. The Mann–Whitney U test was used to compare estradiol distributions between recorded sex groups. The assay detection-limit indicator was inspected to identify observations with estradiol concentrations below the detection threshold. As survey weights, strata and primary sampling units were not incorporated, inferential results were interpreted as exploratory rather than population-representative.


In [ ]:

# Import statistical test

from scipy.stats import mannwhitneyu

# Extract estradiol measurements

male_estradiol = adult_data.loc[
    adult_data["RIAGENDR"] == 1,
    "LBXEST"
].dropna()

female_estradiol = adult_data.loc[
    adult_data["RIAGENDR"] == 2,
    "LBXEST"
].dropna()

# Perform Mann-Whitney U test

u_statistic, p_value = mannwhitneyu(
    male_estradiol,
    female_estradiol,
    alternative="two-sided"
)

# Display results

print("MANN-WHITNEY U TEST")
print("-------------------")

print("U statistic:", u_statistic)
print("P-value:", p_value)

if p_value < 0.05:
    print("Result: Statistically significant difference")
else:
    print("Result: No statistically significant difference")

In [ ]:

# Compare median estradiol concentrations

male_median = male_estradiol.median()
female_median = female_estradiol.median()

median_difference = female_median - male_median

print("MALE MEDIAN:", male_median)
print("FEMALE MEDIAN:", female_median)

print(
    "Female minus male median difference:",
    median_difference
)

In [ ]:

# Investigate estradiol detection-limit indicators

print("ESTRADIOL DETECTION LIMIT ANALYSIS")
print("----------------------------------")

print("\nDetection-limit indicator counts:")

print(
    adult_data["LBDESTLC"].value_counts(
        dropna=False
    )
)

# Compare estradiol measurements by detection status

print("\nEstradiol concentrations by detection status:")

display(
    adult_data.groupby("LBDESTLC")["LBXEST"].agg(
        ["count", "min", "median", "max"]
    )
)

In [ ]:

# Examine the 15 highest estradiol concentrations

highest_estradiol = adult_data[
    ["SEQN", "RIAGENDR", "RIDAGEYR",
     "BMXBMI", "LBXEST", "LBDESTLC"]
].sort_values(
    by="LBXEST",
    ascending=False
).head(15)

print("15 HIGHEST ESTRADIOL MEASUREMENTS")

display(highest_estradiol)

In [ ]:

# Inspect potential hormone predictor columns

potential_predictors = [
    "LBXFSH",
    "LBXLUH",
    "LBXPG4",
    "LBXTST",
    "LBXSHBG"
]

for variable in potential_predictors:
    if variable in adult_data.columns:
        missing = adult_data[variable].isna().sum()
        print(variable, "- Missing values:", missing)
    else:
        print(variable, "- Column not found")

### 2.4 Predictor selection and model development

Eight predictors were selected: recorded sex, age, BMI, follicle-stimulating hormone (FSH), luteinising hormone (LH), progesterone, testosterone and sex hormone-binding globulin (SHBG). Complete-case observations were retained, and estradiol measurements below the assay detection limit were excluded from regression modelling. These selection criteria may affect generalisability.

The response variable was transformed using `log1p` to reduce the influence of the strongly right-skewed concentration distribution. Participants were divided into training (80%) and test (20%) sets, with stratification by recorded sex. A median baseline, Ridge regression, Random Forest and histogram-based Gradient Boosting were fitted. Five-fold cross-validation within the training set was used to compare the tree-based algorithms. Performance was assessed using mean absolute error (MAE) and the coefficient of determination (R²).

### 2.5 Training-set model comparison

Cross-validation scores were calculated separately for Random Forest and Gradient Boosting. The comparison informed selection of an illustrative final model without relying on the repeatedly inspected test set.


In [ ]:

# Select variables for machine learning

model_variables = [
    "RIAGENDR",
    "RIDAGEYR",
    "BMXBMI",
    "LBXFSH",
    "LBXLUH",
    "LBXPG4",
    "LBXTST",
    "LBXSHBG",
    "LBXEST",
    "LBDESTLC"
]

# Create a separate modelling dataset

model_data = adult_data[model_variables].copy()

print("Original adult participants:", len(adult_data))

print("Modelling dataset dimensions:", model_data.shape)

display(model_data.head())

In [ ]:

# Count participants with complete measurements

complete_data = model_data.dropna(
    subset=[
        "RIAGENDR",
        "RIDAGEYR",
        "BMXBMI",
        "LBXFSH",
        "LBXLUH",
        "LBXPG4",
        "LBXTST",
        "LBXSHBG",
        "LBXEST"
    ]
).copy()

print("DATA COMPLETENESS")
print("-----------------")

print("Original adult participants:", len(model_data))

print("Complete observations:", len(complete_data))

print(
    "Participants excluded due to missing values:",
    len(model_data) - len(complete_data)
)

print(
    "Percentage retained:",
    round(len(complete_data) / len(model_data) * 100, 2)
)

In [ ]:

# Check for duplicate participant IDs

print("Duplicate participant IDs:")

print(adult_data["SEQN"].duplicated().sum())

In [ ]:

# Select biological variables

correlation_variables = [
    "RIDAGEYR",
    "BMXBMI",
    "LBXFSH",
    "LBXLUH",
    "LBXPG4",
    "LBXTST",
    "LBXSHBG",
    "LBXEST"
]

# Calculate Spearman correlations

correlation_matrix = adult_data[
    correlation_variables
].corr(method="spearman")

# Create heatmap

plt.figure(figsize=(10, 8))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f"
)

plt.title("Spearman Correlations Between Biological Variables")

plt.tight_layout()

plt.savefig(
    "hormone_correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:

# Create the dataset for the predictive model

# Start with complete observations
ml_data = complete_data.copy()

# Keep estradiol results at or above the detection limit
ml_data = ml_data[
    ml_data["LBDESTLC"] < 0.5
].copy()

print("MACHINE LEARNING DATASET")
print("------------------------")

print("Participants available:", len(ml_data))

print("Missing values:", ml_data.isnull().sum().sum())

In [ ]:

# Select predictor variables

features = [
    "RIAGENDR",
    "RIDAGEYR",
    "BMXBMI",
    "LBXFSH",
    "LBXLUH",
    "LBXPG4",
    "LBXTST",
    "LBXSHBG"
]

# X contains the predictors
X = ml_data[features].copy()

# y contains the estradiol measurements
y = ml_data["LBXEST"].copy()

print("Number of predictors:", X.shape[1])
print("Number of participants:", X.shape[0])

print("\nTarget variable:")
print(y.name)

In [ ]:

# Import the training and testing function

from sklearn.model_selection import train_test_split

# Split the dataset

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=X["RIAGENDR"]
)

print("TRAINING AND TESTING DATA")
print("-------------------------")

print("Training participants:", len(X_train))
print("Testing participants:", len(X_test))

In [ ]:
import numpy as np
# Import machine learning tools

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

# Transform the target to reduce skewness

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# Baseline model
baseline = DummyRegressor(strategy="median")

baseline.fit(X_train, y_train_log)

# Ridge regression model
ridge_model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)

ridge_model.fit(X_train, y_train_log)

print("Both models trained successfully!")

In [ ]:

# Import Random Forest

from sklearn.ensemble import RandomForestRegressor

# Create the model

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

# Train the model using the same training data

rf_model.fit(X_train, y_train_log)

print("Random Forest trained successfully!")

In [ ]:

# Import cross-validation tools

import numpy as np

from sklearn.model_selection import KFold, cross_validate

# Create five cross-validation folds

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluate Random Forest on training data only

cv_results = cross_validate(
    rf_model,
    X_train,
    y_train_log,
    cv=cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    },
    n_jobs=-1
)

# Convert negative MAE scores to positive values

cv_mae = -cv_results["test_mae"]
cv_r2 = cv_results["test_r2"]

print("RANDOM FOREST CROSS-VALIDATION")
print("--------------------------------")

print("MAE for each fold:", np.round(cv_mae, 3))
print("R-squared for each fold:", np.round(cv_r2, 3))

print("\nAverage MAE:", round(cv_mae.mean(), 3))
print("MAE standard deviation:", round(cv_mae.std(), 3))

print("\nAverage R-squared:", round(cv_r2.mean(), 3))
print("R-squared standard deviation:", round(cv_r2.std(), 3))

In [ ]:

# Train Gradient Boosting

from sklearn.ensemble import HistGradientBoostingRegressor

# Create the model

gb_model = HistGradientBoostingRegressor(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=15,
    min_samples_leaf=20,
    random_state=42
)

# Train using our existing training data

gb_model.fit(X_train, y_train_log)

print("Gradient Boosting trained successfully!")

In [ ]:

# Cross-validate Gradient Boosting

from sklearn.model_selection import cross_validate

gb_cv_results = cross_validate(
    gb_model,
    X_train,
    y_train_log,
    cv=cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    },
    n_jobs=-1
)

gb_cv_mae = -gb_cv_results["test_mae"]
gb_cv_r2 = gb_cv_results["test_r2"]

print("GRADIENT BOOSTING CROSS-VALIDATION")
print("----------------------------------")

print("MAE for each fold:", np.round(gb_cv_mae, 3))
print("R-squared for each fold:", np.round(gb_cv_r2, 3))

print("\nAverage MAE:", round(gb_cv_mae.mean(), 3))
print("MAE standard deviation:", round(gb_cv_mae.std(), 3))

print("\nAverage R-squared:", round(gb_cv_r2.mean(), 3))
print("R-squared standard deviation:", round(gb_cv_r2.std(), 3))

In [ ]:

# Compare cross-validation performance

cv_comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Gradient Boosting"
    ],
    "Mean MAE (log scale)": [
        cv_mae.mean(),
        gb_cv_mae.mean()
    ],
    "MAE Standard Deviation": [
        cv_mae.std(),
        gb_cv_mae.std()
    ],
    "Mean R-squared": [
        cv_r2.mean(),
        gb_cv_r2.mean()
    ],
    "R-squared Standard Deviation": [
        cv_r2.std(),
        gb_cv_r2.std()
    ]
})

print("CROSS-VALIDATION MODEL COMPARISON")
print("---------------------------------")

display(cv_comparison.round(3))

## 3. Results

### 3.1 Exploratory test-set model performance

Predictions were generated for the held-out test observations using the fitted baseline, Ridge regression and Random Forest models. Evaluation was undertaken on the log-transformed target and, where indicated, after converting predictions back to pg/mL.

The test split was examined repeatedly during the exploratory development of this project. Consequently, the reported test-set metrics should **not** be interpreted as results from a fully untouched final validation set. Training-set cross-validation provided the primary basis for algorithm comparison.

### 3.2 Feature importance and prediction error

Random Forest feature importance was examined as a descriptive property of the fitted model. Actual and predicted estradiol concentrations were compared graphically, and residuals were summarised by magnitude, direction and observed concentration range. Feature importance measures do not establish biological causation, particularly when predictors are correlated.


In [ ]:

# Generate predictions on the test data

baseline_pred = baseline.predict(X_test)
ridge_pred = ridge_model.predict(X_test)

# Evaluate performance on the log scale

print("MODEL PERFORMANCE (LOG SCALE)")
print("-----------------------------")

for name, predictions in [
    ("Baseline", baseline_pred),
    ("Ridge Regression", ridge_pred)
]:
    mae = mean_absolute_error(y_test_log, predictions)
    r2 = r2_score(y_test_log, predictions)

    print("\n", name)
    print("MAE:", round(mae, 3))
    print("R-squared:", round(r2, 3))

# Convert predictions back to pg/mL

ridge_pred_pg = np.expm1(ridge_pred)

print("\nRIDGE REGRESSION (ORIGINAL SCALE)")
print("--------------------------------")

print(
    "MAE (pg/mL):",
    round(mean_absolute_error(y_test, ridge_pred_pg), 2)
)

In [ ]:

# Generate Random Forest predictions

rf_pred_log = rf_model.predict(X_test)

# Calculate performance on log scale

rf_mae = mean_absolute_error(
    y_test_log,
    rf_pred_log
)

rf_r2 = r2_score(
    y_test_log,
    rf_pred_log
)

# Convert predictions back to pg/mL

rf_pred_pg = np.expm1(rf_pred_log)

rf_mae_pg = mean_absolute_error(
    y_test,
    rf_pred_pg
)

print("RANDOM FOREST PERFORMANCE")
print("-------------------------")

print("MAE (log scale):", round(rf_mae, 3))
print("R-squared (log scale):", round(rf_r2, 3))
print("MAE (pg/mL):", round(rf_mae_pg, 2))

In [ ]:

# Compare model performance

import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "Baseline",
        "Ridge Regression",
        "Random Forest"
    ],
    "MAE (log scale)": [
        mean_absolute_error(y_test_log, baseline_pred),
        mean_absolute_error(y_test_log, ridge_pred),
        rf_mae
    ],
    "R-squared (log scale)": [
        r2_score(y_test_log, baseline_pred),
        r2_score(y_test_log, ridge_pred),
        rf_r2
    ]
})

display(comparison.round(3))

In [ ]:

# Calculate Random Forest feature importance

importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("RANDOM FOREST FEATURE IMPORTANCE")

display(importance.round(3))

# Visualise feature importance

plt.figure(figsize=(9, 6))

sns.barplot(
    data=importance,
    x="Importance",
    y="Feature"
)

plt.title("Random Forest Feature Importance")
plt.xlabel("Relative importance")
plt.ylabel("Predictor")

plt.tight_layout()

plt.savefig(
    "random_forest_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:

# Plot actual versus predicted estradiol

plt.figure(figsize=(8, 7))

plt.scatter(
    y_test,
    rf_pred_pg,
    alpha=0.4,
    s=20
)

# Reference line for perfect predictions

max_value = max(y_test.max(), rf_pred_pg.max())

plt.plot(
    [0, max_value],
    [0, max_value],
    linestyle="--",
    color="red",
    label="Perfect prediction"
)

plt.xscale("log")
plt.yscale("log")

plt.xlabel("Actual estradiol (pg/mL)")
plt.ylabel("Predicted estradiol (pg/mL)")

plt.title("Random Forest: Actual vs Predicted Estradiol")

plt.legend()
plt.tight_layout()

plt.savefig(
    "random_forest_predictions.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:

# Calculate prediction errors

import pandas as pd
import numpy as np

# Create a table containing actual and predicted values

error_data = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": rf_pred_pg
})

# Calculate prediction errors

error_data["Error"] = (
    error_data["Predicted"] - error_data["Actual"]
)

error_data["Absolute_Error"] = (
    error_data["Error"].abs()
)

# Calculate absolute percentage error
# Only for positive actual concentrations

error_data["Percentage_Error"] = (
    error_data["Absolute_Error"]
    / error_data["Actual"]
) * 100

print("PREDICTION ERROR SUMMARY")
print("------------------------")

display(error_data.describe().round(2))

In [ ]:

# Find the largest prediction errors

largest_errors = error_data.sort_values(
    by="Absolute_Error",
    ascending=False
).head(10)

print("10 LARGEST PREDICTION ERRORS")
print("----------------------------")

display(largest_errors.round(2))

In [ ]:

# Compare errors by concentration range

# Define concentration categories

bins = [0, 10, 50, 200, np.inf]

labels = [
    "Low (<10)",
    "Moderate (10-<50)",
    "High (50-<200)",
    "Very high (200+)"
]

# Categorise participants

error_data["Concentration_Group"] = pd.cut(
    error_data["Actual"],
    bins=bins,
    labels=labels,
    right=False
)

# Calculate errors for each group

group_errors = error_data.groupby(
    "Concentration_Group",
    observed=True
).agg(
    Participants=("Actual", "count"),
    Mean_Absolute_Error=("Absolute_Error", "mean"),
    Median_Absolute_Error=("Absolute_Error", "median")
)

print("MODEL ACCURACY BY ESTRADIOL CONCENTRATION")
print("-----------------------------------------")

display(group_errors.round(2))

In [ ]:

# Visualise prediction errors

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=error_data,
    x="Concentration_Group",
    y="Absolute_Error",
    showfliers=False
)

plt.yscale("log")

plt.title(
    "Random Forest Prediction Errors by Estradiol Concentration"
)

plt.xlabel("Actual estradiol concentration (pg/mL)")
plt.ylabel("Absolute prediction error (pg/mL, log scale)")

plt.xticks(rotation=15)

plt.tight_layout()

plt.savefig(
    "prediction_errors_by_concentration.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:

# Calculate prediction bias

print("PREDICTION BIAS")
print("---------------")

print(
    "Mean prediction error:",
    round(error_data["Error"].mean(), 2),
    "pg/mL"
)

print(
    "Median prediction error:",
    round(error_data["Error"].median(), 2),
    "pg/mL"
)

print(
    "Percentage of predictions below actual values:",
    round((error_data["Error"] < 0).mean() * 100, 2),
    "%"
)

### 3.3 Pregnancy status and extreme estradiol concentrations

Pregnancy status (`RIDEXPRG`) was linked from the NHANES demographics file to investigate the biological context of extreme estradiol measurements and large prediction errors. Status codes distinguished participants recorded as pregnant, not pregnant or of indeterminate status. Missing or inapplicable records were not treated as evidence of non-pregnancy.

Prediction errors were compared descriptively among test-set participants with recorded pregnancy status. These comparisons were interpreted cautiously because subgroup sizes were small and observed estradiol concentrations differed markedly between groups.


In [ ]:

# Add pregnancy information

# Create a separate dataset so we don't overwrite adult_data

pregnancy_data = adult_data.merge(
    demographics[["SEQN", "RIDEXPRG"]],
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("PREGNANCY DATA MERGED")
print("---------------------")

print("Original participants:", len(adult_data))
print("Participants after merge:", len(pregnancy_data))

print("\nPregnancy status counts:")

print(
    pregnancy_data["RIDEXPRG"].value_counts(
        dropna=False
    )
)

In [ ]:

# Investigate high estradiol concentrations

highest_estradiol_pregnancy = pregnancy_data[
    [
        "SEQN",
        "RIAGENDR",
        "RIDAGEYR",
        "LBXEST",
        "RIDEXPRG"
    ]
].sort_values(
    by="LBXEST",
    ascending=False
).head(20)

print("20 HIGHEST ESTRADIOL MEASUREMENTS")
print("--------------------------------")

display(highest_estradiol_pregnancy)

In [ ]:

# Match prediction errors with pregnancy status

# Retrieve pregnancy status for each test participant
# using the original participant ID (SEQN)

test_pregnancy = pregnancy_data.set_index(
    "SEQN"
)["RIDEXPRG"]

# Retrieve SEQN values corresponding to the test observations
test_ids = adult_data.loc[X_test.index, "SEQN"]

# Create a new error table
pregnancy_errors = error_data.copy()

# Add participant IDs
pregnancy_errors["SEQN"] = test_ids.to_numpy()

# Match pregnancy status by participant ID
pregnancy_errors["Pregnancy_Status"] = (
    pregnancy_errors["SEQN"].map(test_pregnancy)
)

print("PREGNANCY STATUS IN TEST DATA")
print("-----------------------------")

print(
    pregnancy_errors["Pregnancy_Status"].value_counts(
        dropna=False
    )
)

In [ ]:

# Compare errors by pregnancy status

# Select participants with known pregnancy status

known_pregnancy = pregnancy_errors[
    pregnancy_errors["Pregnancy_Status"].isin([1, 2])
].copy()

# Convert numerical codes to readable labels

known_pregnancy["Pregnancy_Group"] = (
    known_pregnancy["Pregnancy_Status"].map({
        1: "Pregnant",
        2: "Not pregnant"
    })
)

# Calculate prediction errors for each group

pregnancy_comparison = known_pregnancy.groupby(
    "Pregnancy_Group"
).agg(
    Participants=("Actual", "count"),
    Mean_Absolute_Error=("Absolute_Error", "mean"),
    Median_Absolute_Error=("Absolute_Error", "median"),
    Mean_Actual_Estradiol=("Actual", "mean")
)

print("MODEL ACCURACY BY PREGNANCY STATUS")
print("----------------------------------")

display(pregnancy_comparison.round(2))

In [ ]:

# Largest errors and pregnancy status

print("15 LARGEST ERRORS WITH PREGNANCY STATUS")
print("---------------------------------------")

display(
    pregnancy_errors[
        [
            "SEQN",
            "Actual",
            "Predicted",
            "Absolute_Error",
            "Pregnancy_Status"
        ]
    ].sort_values(
        by="Absolute_Error",
        ascending=False
    ).head(15).round(2)
)

### 3.4 Final model metrics and concentration-based sensitivity analysis

The selected Random Forest model was evaluated using both log-transformed and original-scale outcomes. To assess sensitivity to extreme concentrations, prediction errors and R² were calculated within observed estradiol ranges and after restricting the evaluation sample to concentrations below 200 pg/mL.

The concentration-defined groups were constructed retrospectively from measured outcomes and do not represent groups that could be assigned prospectively without an estradiol measurement. Negative subgroup R² values were possible because the variability of observed estradiol within narrow ranges was limited.


In [ ]:

# Final Random Forest evaluation

from sklearn.metrics import mean_absolute_error, r2_score

# Predict estradiol on the log scale
final_predictions_log = rf_model.predict(X_test)

# Convert predictions back to pg/mL
final_predictions_pg = np.expm1(final_predictions_log)

# Calculate log-scale performance
final_mae_log = mean_absolute_error(
    y_test_log,
    final_predictions_log
)

final_r2_log = r2_score(
    y_test_log,
    final_predictions_log
)

# Calculate original-scale performance
final_mae_pg = mean_absolute_error(
    y_test,
    final_predictions_pg
)

# Calculate original-scale R-squared
final_r2_pg = r2_score(
    y_test,
    final_predictions_pg
)

print("FINAL RANDOM FOREST PERFORMANCE")
print("-------------------------------")

print("Training participants:", len(X_train))
print("Testing participants:", len(X_test))

print("\nLOG-TRANSFORMED ESTRADIOL")
print("MAE:", round(final_mae_log, 3))
print("R-squared:", round(final_r2_log, 3))

print("\nORIGINAL ESTRADIOL (pg/mL)")
print("MAE:", round(final_mae_pg, 2))
print("R-squared:", round(final_r2_pg, 3))

In [ ]:

# Create final model summary

final_results = pd.DataFrame({
    "Metric": [
        "Training participants",
        "Testing participants",
        "Cross-validation MAE (log)",
        "Cross-validation R-squared",
        "Test MAE (log)",
        "Test R-squared (log)",
        "Test MAE (pg/mL)",
        "Test R-squared (pg/mL)"
    ],
    "Value": [
        len(X_train),
        len(X_test),
        round(cv_mae.mean(), 3),
        round(cv_r2.mean(), 3),
        round(final_mae_log, 3),
        round(final_r2_log, 3),
        round(final_mae_pg, 2),
        round(final_r2_pg, 3)
    ]
})

display(final_results)

# Export results for the GitHub project
final_results.to_csv(
    "random_forest_results.csv",
    index=False
)

print("Results saved successfully!")

In [ ]:

# Model performance by estradiol range

from sklearn.metrics import mean_absolute_error, r2_score

# Create a table containing actual and predicted values

range_results = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": final_predictions_pg
})

# Define estradiol concentration groups

range_results["Estradiol_Range"] = pd.cut(
    range_results["Actual"],
    bins=[0, 10, 50, 200, float("inf")],
    labels=[
        "Below 10",
        "10 to <50",
        "50 to <200",
        "200 and above"
    ],
    right=False
)

# Calculate performance within each group

summary_rows = []

for group, data in range_results.groupby(
    "Estradiol_Range",
    observed=True
):

    mae = mean_absolute_error(
        data["Actual"],
        data["Predicted"]
    )

    r2 = (
        r2_score(data["Actual"], data["Predicted"])
        if len(data) >= 2 else np.nan
    )

    summary_rows.append({
        "Estradiol range": group,
        "Participants": len(data),
        "MAE (pg/mL)": round(mae, 2),
        "R-squared": round(r2, 3)
    })

range_performance = pd.DataFrame(summary_rows)

print("MODEL PERFORMANCE BY ESTRADIOL RANGE")
print("------------------------------------")

display(range_performance)

In [ ]:

# Sensitivity analysis

# Select test participants with estradiol below 200 pg/mL

lower_range = range_results[
    range_results["Actual"] < 200
]

# Calculate model performance

lower_mae = mean_absolute_error(
    lower_range["Actual"],
    lower_range["Predicted"]
)

lower_r2 = r2_score(
    lower_range["Actual"],
    lower_range["Predicted"]
)

print("SENSITIVITY ANALYSIS: ESTRADIOL BELOW 200 pg/mL")
print("------------------------------------------------")

print("Participants:", len(lower_range))
print("MAE:", round(lower_mae, 2), "pg/mL")
print("R-squared:", round(lower_r2, 3))

print("\nFULL TEST SET")
print("-------------")

print("Participants:", len(y_test))
print("MAE:", round(final_mae_pg, 2), "pg/mL")
print("R-squared:", round(final_r2_pg, 3))

## 4. Discussion

### 4.1 Comparison of regression models

Random Forest and histogram-based Gradient Boosting produced similar performance during five-fold cross-validation of the training data. Random Forest achieved a mean R² of **0.788** and a mean MAE of **0.377** on the log-transformed scale, compared with **0.790** and **0.379**, respectively, for Gradient Boosting. The differences were small and did not establish a clear advantage for either method. Random Forest was retained as the illustrative final model because its cross-validated MAE was marginally lower and its predictions had been investigated in detail.

On the exploratory test set of 1,027 observations, Random Forest achieved a log-scale R² of **0.741**. Following inverse transformation, original-scale R² was **0.899** and MAE was **17.24 pg/mL**. The relatively high original-scale R² requires careful interpretation because extreme estradiol values contribute disproportionately to total outcome variance.

### 4.2 Variation in predictive performance

Prediction errors increased with measured estradiol concentration. Mean absolute error ranged from **3.12 pg/mL** for observations below 10 pg/mL to **195.64 pg/mL** for those at or above 200 pg/mL. Although the model achieved an original-scale R² of 0.899 across the full test set, R² decreased to **0.677** when evaluation was restricted to the 989 observations below 200 pg/mL; MAE in this restricted group was **10.39 pg/mL**. This sensitivity analysis indicates that the overall R² was strongly influenced by the upper end of the concentration distribution.

The negative R² values obtained for several narrower concentration groups indicate poorer performance than a within-group mean benchmark under that metric. These values should be considered alongside absolute error, because the restricted variation in measured concentrations affects the interpretation of R².

### 4.3 Biological interpretation

FSH and progesterone contributed prominently to Random Forest feature importance, consistent with their relevance to reproductive endocrine physiology. Nevertheless, impurity-based importance can be affected by correlations among predictors and should not be interpreted as a causal estimate.

All 20 participants with the highest observed estradiol concentrations were recorded as pregnant. In the test set, the six participants recorded as pregnant had a mean absolute error of **299.20 pg/mL**, compared with **43.07 pg/mL** among 181 participants recorded as not pregnant. Mean estradiol concentrations also differed substantially between these groups. The observed difference in absolute prediction error therefore cannot be attributed independently to pregnancy, and the small pregnant subgroup precludes firm conclusions about predictive performance in pregnancy.

### 4.4 Limitations

Several limitations affect interpretation of the findings. First, NHANES uses a complex sampling design, but survey weights, stratification and clustering were not incorporated. The descriptive and inferential results therefore should not be generalised as nationally representative estimates.

Second, complete-case selection and exclusion of estradiol measurements below the assay detection limit may have introduced selection bias. Third, pregnancy status was missing or inapplicable for many participants and was not included as a model predictor. Menstrual cycle phase, hormonal medication use and other potentially relevant physiological or clinical factors were not explicitly modelled.

Fourth, the predictors included other laboratory-measured hormones. Accordingly, the model does not provide a practical substitute for estradiol laboratory testing. In addition, correlated predictors complicate the interpretation of feature importance.

Finally, the test set was repeatedly examined during exploratory analysis and no independent external validation was performed. The reported model performance should therefore be regarded as exploratory rather than evidence of clinical validity.

## 5. Conclusion

The analysis demonstrated that demographic, anthropometric and concurrent reproductive hormone measurements contained predictive information about serum estradiol concentration in the selected NHANES cohort. Random Forest and Gradient Boosting showed comparable cross-validation performance. However, concentration-specific error analysis and pregnancy-related observations demonstrated that a high overall R² can obscure substantial heterogeneity in predictive accuracy.

The findings illustrate the value of integrating statistical model evaluation with biological interpretation. Further work would require more rigorous validation, consideration of the NHANES sampling design and assessment of additional physiological covariates before any clinical application could be considered.
